# A Kaggle pipeline is a graph

Written as a list of steps, a modelling pipeline hides its own structure. Load,
split, clean, encode, fit, evaluate reads as one thing after another — and it is
not. Numeric and categorical encoding do not depend on each other. They run at
the same time, they fail independently, and they meet at one place where the
feature matrix is assembled.

That meeting point is a **join**, and a join is the thing a sequence cannot
express. Everything in this notebook follows from taking it seriously.

Nothing here downloads a dataset or fits a model. The point is the *structure* —
what can be checked before anything expensive runs.

In [1]:
# Installed from the repository, not from PyPI: this notebook uses `templates`
# and `viz`, which no published release contains yet. A notebook that installs
# something older than the API it calls fails at cell one, which is a confusing
# way to introduce a library about checking things before they run.
try:
    import browsergraph  # noqa: F401
except ImportError:  # pragma: no cover
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import browsergraph as bg
from browsergraph import templates as T, viz
from browsergraph.compile import CompileError, compile_route
from browsergraph.manifest import NodeManifest, ParameterSpec, PortSpec
from browsergraph.workbench import NodeCandidate

print("browsergraph", bg.__version__)

browsergraph 0.3.0


In [2]:
# From the library, not redefined here. A notebook that teaches helpers
# browsergraph does not have is a notebook nobody can build on.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step

## Start from a template, not a blank page

The steps of a supervised tabular problem are not a research question. They have
a stable, boring answer, and re-deriving it per project is how two pipelines end
up incomparable. A template is that answer as a typed skeleton: every port
declared, every slot empty.

In [3]:
template = T.get("tabular.supervised")

print(template.task, "\n")
for slot in template.slots:
    ports_in = ", ".join(f"{n}:{t}" for n, t in slot.inputs) or "—"
    ports_out = ", ".join(f"{n}:{t}" for n, t in slot.outputs)
    flag = "  (optional)" if slot.optional else ""
    print(f"  {slot.id:<12} {ports_in:>28}  ->  {ports_out}{flag}")

Fit a model on a tabular dataset and produce scored predictions. 

  load                                    —  ->  out:Frame
  split                            in:Frame  ->  train:Frame, valid:Frame
  clean                            in:Frame  ->  out:Frame
  numeric                          in:Frame  ->  out:Matrix
  categorical                      in:Frame  ->  out:Matrix
  assemble     numeric:Matrix, categorical:Matrix  ->  out:Matrix
  fit                             in:Matrix  ->  out:Model
  calibrate                        in:Model  ->  out:Model  (optional)
  evaluate                         in:Model  ->  out:Score


In [4]:
skeleton = template.skeleton()
print("layers:", skeleton.layers())
print("is a chain:", skeleton.is_chain)

layers: [['load'], ['split'], ['clean'], ['numeric', 'categorical'], ['assemble'], ['fit'], ['calibrate'], ['evaluate']]
is a chain: False


`['numeric', 'categorical']` arriving as one layer is the whole claim, in
output form. Those two steps are independent, and the model says so without
anyone having asserted it — it is derived from which ports feed which.

## The anti-patterns travel with the shape

Guidance in a document is guidance nobody loads. These come attached to the
template, so a harness holding the shape is holding the warnings too.

In [5]:
for i, warning in enumerate(template.anti_patterns, 1):
    print(f"{i}. {warning}\n")

1. Fitting the encoder on the full frame before splitting. The leak is invisible in cross-validation and fatal on the held-out set — this is the single most common way a good-looking pipeline is wrong.

2. Treating 'more preprocessing steps' as progress. Each step is a place to leak and a thing to maintain; the count is not the score.

3. Ensembling models that share a failure mode. Three gradient-boosted trees on the same features are one model with more variance.

4. Reporting the best validation score seen across many attempts as if it were an estimate of future performance. It is the maximum of a sample and biased upward by exactly the amount of searching you did.



## Fill the slots

A slot says *what contract this step satisfies*. A candidate says *how*. Below,
several ways to do each step — which is what turns one pipeline into a space of
them.

In [6]:
nodes = [
    node("tab.load.csv",        "data.read",       [], [("out", "Frame")]),
    node("tab.load.parquet",    "data.read",       [], [("out", "Frame")]),

    node("tab.split.holdout",   "data.split",      [("in", "Frame")],
         [("train", "Frame"), ("valid", "Frame")]),
    node("tab.split.kfold",     "data.split",      [("in", "Frame")],
         [("train", "Frame"), ("valid", "Frame")],
         parameters=[ParameterSpec("k", "int", choices=(5, 10), default=5)]),

    node("tab.clean.median",    "data.clean",      [("in", "Frame")], [("out", "Frame")]),
    node("tab.clean.drop",      "data.clean",      [("in", "Frame")], [("out", "Frame")]),

    node("tab.num.standard",    "feature.numeric", [("in", "Frame")], [("out", "Matrix")]),
    node("tab.num.quantile",    "feature.numeric", [("in", "Frame")], [("out", "Matrix")]),

    node("tab.cat.onehot",      "feature.categorical", [("in", "Frame")], [("out", "Matrix")]),
    node("tab.cat.target",      "feature.categorical", [("in", "Frame")], [("out", "Matrix")],
         facets={"purpose.not_for": ["high-cardinality with tiny folds"],
                 "failure.modes": "leaks the target if fitted before the split"}),

    node("tab.assemble.hstack", "feature.assemble",
         [("numeric", "Matrix"), ("categorical", "Matrix")], [("out", "Matrix")]),

    node("tab.fit.gbm",         "model.fit",       [("in", "Matrix")], [("out", "Model")],
         deterministic=False),
    node("tab.fit.linear",      "model.fit",       [("in", "Matrix")], [("out", "Model")]),

    node("tab.cal.isotonic",    "model.calibrate", [("in", "Model")], [("out", "Model")]),

    node("tab.eval.auc",        "model.evaluate",  [("in", "Model")], [("out", "Score")]),
    node("tab.eval.logloss",    "model.evaluate",  [("in", "Model")], [("out", "Score")]),
]

filling = {
    "load":        ["tab.load.csv", "tab.load.parquet"],
    "split":       ["tab.split.holdout", "tab.split.kfold"],
    "clean":       ["tab.clean.median", "tab.clean.drop"],
    "numeric":     ["tab.num.standard", "tab.num.quantile"],
    "categorical": ["tab.cat.onehot", "tab.cat.target"],
    "assemble":    ["tab.assemble.hstack"],
    "fit":         ["tab.fit.gbm", "tab.fit.linear"],
    "calibrate":   ["tab.cal.isotonic"],
    "evaluate":    ["tab.eval.auc", "tab.eval.logloss"],
}

bench = template.instantiate(filling)
bench = bench.__class__(**{**bench.__dict__, "nodes": tuple(nodes)})

print("still unfilled:", template.unfilled(filling) or "nothing")
print("complete routes:", f"{bench.route_count():,}")

still unfilled: nothing
complete routes: 128


## Look at it

In [7]:
viz.dag(bench, route={"numeric": "tab.num.standard"})

Figure(svg='<svg viewBox="0 0 1590 308" width="1590" height="308" style="max-width:none" role="img"><defs><marker id="bg5914239-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Load dataset</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Split</text><text x="279" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="480" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Clean</text><text x="489" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3 · 2 parallel</text><g><rect x="690" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#c0392b" stroke-width="2"/><text x="699" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Encode numeric</text><text x="699" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text><text x="699" y="120.0" font-size="9.5" fill="#c0392b">standard</text></g><g><rect x="690" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Encode categorical</text><text x="699" y="190.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Assemble features</text><text x="909" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1203.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 5</text><g><rect x="1110" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1119" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Fit model</text><text x="1119" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="1413.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 6</text><g><rect x="1320" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1329" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Calibrate</text><text x="1329" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1623.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 7</text><g><rect x="1530" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1539" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Evaluate</text><text x="1539" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><path d="M246,141.0 C258.0,141.0 258.0,141.0 270,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg5914239-arrow)"/><path d="M456,141.0 C468.0,141.0 468.0,141.0 480,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg5914239-arrow)"/><text x="468.0" y="136.0" text-anchor="middle" font-size="9" fill="#68737f">train</text><path d="M666,141.0 C

Two boxes in layer 3 — the parallel encoders — and one box in layer 4 with two
inbound arrows labelled with the ports they land on. That picture is the
difference between a pipeline and a graph, and it is derived, not drawn by hand.

In [8]:
viz.route_space(bench, max_rows=4)

Figure(svg='<svg viewBox="0 0 1480 230" width="1480" height="230" style="max-width:none" role="img"><style>.bg84348783-v{cursor:pointer}.bg84348783-v:hover rect{stroke:#c0392b;stroke-width:2}</style><polyline points="46,96 170,96 204,96 328,96 362,96 486,96 520,96 644,96 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 204,96 328,96 362,96 486,96 520,96 644,96 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 204,126 328,126 362,96 486,96 520,96 644,96 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 204,126 328,126 362,96 486,96 520,96 644,96 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 204,96 328,96 362,126 486,126 520,96 644,96 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 204,96 328,96 362,126 486,126 520,96 644,96 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 204,126 328,126 362,126 486,126 520,96 644,96 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 204,126 328,126 362,126 486,126 520,96 644,96 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 204,96 328,96 362,96 486,96 520,126 644,126 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 204,96 328,96 362,96 486,96 520,126 644,126 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 204,126 328,126 362,96 486,96 520,126 644,126 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 204,126 328,126 362,96 486,96 520,126 644,126 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 204,96 328,96 362,126 486,126 520,126 644,126 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 204,96 328,96 362,126 486,126 520,126 644,126 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 204,126 328,126 362,126 486,126 520,126 644,126 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 204,126 328,126 362,126 486,126 520,126 644,126 678,96 802,96 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 204,96 328,96 362,96 486,96 520,96 644,96 678,126 802,126 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 204,96 328,96 362,96 486,96 520,96 644,96 678,126 802,126 836,96 960,96 994,96 1118,96 1152,96 1276,96 1310,96 1434,96" fill="none" 

## Compile one route

Compiling resolves a route into a frozen plan: every port checked against the
*chosen* candidates rather than the stage declarations, permissions and effects
gathered, and a content hash over the whole thing so two runs can be compared.

In [9]:
route = {"load": "tab.load.csv", "split": "tab.split.holdout",
         "clean": "tab.clean.median", "numeric": "tab.num.standard",
         "categorical": "tab.cat.onehot", "assemble": "tab.assemble.hstack",
         "fit": "tab.fit.linear", "calibrate": "tab.cal.isotonic",
         "evaluate": "tab.eval.auc"}

plan = compile_route(bench, route)
print(plan.digest)
print("layers      :", plan.layers)
print("parallel    :", plan.parallel_width, "steps may run at once")
print("deterministic:", plan.deterministic)

plan:fb6962331eb7a115911090af53ee28fe
layers      : (('load',), ('split',), ('clean',), ('numeric', 'categorical'), ('assemble',), ('fit',), ('calibrate',), ('evaluate',))
parallel    : 2 steps may run at once
deterministic: True


Swap the linear model for the boosted one and the plan stops claiming to be
deterministic — because that node declared it is not. Nobody had to remember.

In [10]:
gbm = compile_route(bench, {**route, "fit": "tab.fit.gbm"})
print("deterministic:", gbm.deterministic)
print("digest changed:", gbm.digest != plan.digest)

deterministic: False
digest changed: True


## The check that earns its keep

A candidate whose output type does not match what the next slot consumes is
refused at compile time, with the edge and the reason. This is the failure that
otherwise surfaces forty minutes into a fit.

In [11]:
broken = node("tab.num.wrong", "feature.numeric",
              [("in", "Frame")], [("out", "Frame")])   # Frame, not Matrix
sabotaged = bench.__class__(**{**bench.__dict__,
                               "nodes": bench.nodes + (broken,),
                               "candidates": bench.candidates
                                             + (NodeCandidate(id="tab.num.wrong",
                                                              node_id="tab.num.wrong"),)})
stages = tuple(
    s.__class__(**{**s.__dict__, "candidates": s.candidates + ("tab.num.wrong",)})
    if s.id == "numeric" else s for s in sabotaged.stages)
sabotaged = sabotaged.__class__(**{**sabotaged.__dict__, "stages": stages})

try:
    compile_route(sabotaged, {**route, "numeric": "tab.num.wrong"})
except CompileError as exc:
    for problem in exc.problems:
        print("refused:", problem)

refused: numeric: 'tab.num.wrong' does not produce 'Matrix' on port 'out' — it gives ['Frame']


## Descriptors rank, they never bind

`tab.cat.target` carries prose about where it is the wrong tool and how it fails.
None of that changes whether it compiles — it changes which legal candidate a
searcher should prefer. Verify that directly:

In [12]:
from dataclasses import replace

described = tuple(replace(n, facets={**n.facets, "quality.prior": 0.9,
                                     "purpose.statement": "encode a column"})
                  for n in bench.nodes)
same = compile_route(replace(bench, nodes=described), route)
print("plan digest unchanged after describing every node:",
      same.digest == plan.digest)

for name, (text, weight) in bg.facet_fields(
        bench.nodes_by_id["tab.cat.target"].facets).items():
    print(f"  {name:<22} w={weight}  {text}")

plan digest unchanged after describing every node: True
  purpose.not_for        w=1.0  high-cardinality with tiny folds
  failure.modes          w=1.0  leaks the target if fitted before the split


## How much of the space did we look at?

In [13]:
viz.funnel([
    ("all routes",      bench.route_count()),
    ("type-legal",      256),
    ("policy-eligible", 128),
    ("evaluated",       12),
    ("chosen",          1),
], title="tabular pipeline — search space")

Figure(svg='<svg viewBox="0 0 1000 342" width="1000" height="342" style="max-width:none" role="img"><text x="176" y="83" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">all routes</text><rect x="190" y="66" width="586.8" height="26" rx="4" fill="#2d6cb5" opacity="0.66" stroke="#2d6cb5" stroke-width="1"/><text x="786.8" y="83" font-size="11" fill="#22303f">128</text><text x="176" y="129" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">type-legal</text><rect x="190" y="112" width="670.0" height="26" rx="4" fill="#2d6cb5" opacity="0.72" stroke="#2d6cb5" stroke-width="1"/><text x="870.0" y="129" font-size="11" fill="#22303f">256</text><text x="934.0" y="129" font-size="10" fill="#68737f">÷0.5</text><text x="176" y="175" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">policy-eligible</text><rect x="190" y="158" width="586.8" height="26" rx="4" fill="#2d6cb5" opacity="0.66" stroke="#2d6cb5" stroke-width="1"/><text x="786.8" y="175" font-size="11" fill="#22303f">128</text><text x="850.8" y="175" font-size="10" fill="#68737f">÷2</text><text x="176" y="221" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">evaluated</text><rect x="190" y="204" width="309.7" height="26" rx="4" fill="#2d6cb5" opacity="0.45" stroke="#2d6cb5" stroke-width="1"/><text x="509.7" y="221" font-size="11" fill="#22303f">12</text><text x="573.7" y="221" font-size="10" fill="#68737f">÷10.7</text><text x="176" y="267" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">chosen</text><rect x="190" y="250" width="83.7" height="26" rx="4" fill="#1f8a4c" opacity="0.28" stroke="#1f8a4c" stroke-width="1"/><text x="283.7" y="267" font-size="11" fill="#22303f">1</text><text x="347.7" y="267" font-size="10" fill="#68737f">÷12</text><text x="190" y="324" font-size="9.5" fill="#68737f">bar length is log-scaled; labels are exact counts</text></svg>', title='tabular pipeline — search space', note='Every row is a real filter, in order.', width=1000, height=342)

## What this bought

* The parallel structure was **derived** from port wiring, not asserted.
* A type error was caught **before** anything was fitted.
* Determinism was **inherited** from the chosen node rather than remembered.
* The plan has a **digest**, so a result can be attributed to an exact graph.
* Descriptions were provably **unable** to affect any of the above.

None of that is specific to tabular data. The next notebook does the same thing
to a document, and the one after that to a system with no data science in it.